In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# C2 2A: raw YUV420p-only diagnostic

This notebook reads the fixed RGB8 three-arm package and isolates RGB8 → rawvideo YUV420p → RGB8. It reuses the original FFmpeg RGB24 geometry, frame rate, default conversion and YUV420p sampling, but deliberately omits libx264 and the MP4 container. It performs no terminal generation, transformer loading, VAE decode, CRF scan, or H.264 encode.

In [ ]:
from pathlib import Path
import sys, subprocess
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH = 'c2a-2a-colab-preparation'
SOURCE = Path('/content/c2a_yuv420_diagnostic_source')
if SOURCE.exists():
    raise FileExistsError('Use a fresh runtime; preserve existing source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_BRANCH], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
print('Source:', subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True)
# Keep Colab CUDA PyTorch; actual versions are recorded by the result.


## Fixed source and fixed work

The only input is `MyDrive/Video-WM/C2A_Quantization_Diagnostic/c2a_quantization_20260915T005839Z`, using its saved ZERO/PLUS_E2/MINUS_E2 RGB8 tensors. The raw conversion keeps RGB24, 320x512 geometry, 49 frames, 8 fps, FFmpeg defaults and yuv420p. A rawvideo stream does not contain the MP4/H.264 colour metadata or quantization, so it cannot strictly reproduce the original MP4 output; the result records both conversion commands and this boundary. Expected work is one frozen FP32 VAE load, three VAE encodes, three raw-YUV encodes, and three raw-YUV decodes; generation/transformer/VAE decode are zero.

In [ ]:
from datetime import datetime, timezone
CONFIG = SOURCE / 'runtime/c2a/c2a_yuv420_diagnostic_run.json'
RUN_ID = 'c2a_yuv420_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/C2A_YUV420_Diagnostic') / RUN_ID
print(CONFIG.read_text())
print('Output:', OUTPUT)
if OUTPUT.exists():
    raise FileExistsError(str(OUTPUT))


In [ ]:
import os, signal
command = [sys.executable, '-m', 'runtime.c2a.run_yuv420_diagnostic', '--config', str(CONFIG), '--output', str(OUTPUT), '--execute']
process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True)
try:
    returncode = process.wait()
except BaseException:
    try: process.send_signal(signal.SIGTERM)
    except ProcessLookupError: pass
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try: os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError: pass
        process.wait()
    raise
print('launcher exit', returncode)
print((OUTPUT / 'result.json').read_text() if (OUTPUT / 'result.json').exists() else 'No result file')
if returncode: raise subprocess.CalledProcessError(returncode, command)


## Persisted result

The enabled run writes only to `MyDrive/Video-WM/C2A_YUV420_Diagnostic/<UTC-run-id>/`. It saves raw YUV420p intermediates, RGB8 roundtrip tensors, raw conversion commands/default-parameter boundary, source/RGB8/raw-YUV/source-MP4 q, direct comparisons, O2/M2 vectors/norms, calls and retained failures. Pixel error remains explicitly non-q evidence.